# DiT and Flow Matching

Two paradigm shifts in generative modeling:

1. **U-Net → Transformer (DiT)** -- Transformers scale better, have simpler architecture (no skip connections, no up/downsampling), and benefit from proven scaling laws. Patchification replaces spatial hierarchy.

2. **DDPM → Flow Matching** -- Linear interpolation instead of noise schedules, predict velocity instead of noise, straighter sampling trajectories = fewer steps needed. This is why SD3 and Flux use flow matching.

**Time:** ~3 hrs | **Hardware:** Needs GPU for training (RTX 2080+ recommended)

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
from torchvision.utils import make_grid
import matplotlib.pyplot as plt
import numpy as np
from tqdm.auto import tqdm
from einops import rearrange

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")
if device.type == "cuda":
    print(f"GPU: {torch.cuda.get_device_name()}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")

In [ ]:
# GPU: ~6 GB VRAM for loading DiT-XL/2 in fp16
# Load DiT-XL/2 and inspect architecture
from diffusers import DiTPipeline, DPMSolverMultistepScheduler

try:
    pipe = DiTPipeline.from_pretrained(
        "facebook/DiT-XL-2-256",
        torch_dtype=torch.float16,
    )
    pipe.scheduler = DPMSolverMultistepScheduler.from_config(pipe.scheduler.config)
    pipe.to(device)
    dit_loaded = True
except Exception as e:
    print(f"Could not load DiT-XL/2 (need ~6GB VRAM): {e}")
    print("Continuing with architecture analysis from documentation.")
    dit_loaded = False

if dit_loaded:
    transformer = pipe.transformer
    print("=" * 80)
    print("DiT-XL/2 Architecture Tree")
    print("=" * 80)

    # Top-level components
    for name, child in transformer.named_children():
        num_params = sum(p.numel() for p in child.parameters()) / 1e6
        print(f"\n{'─' * 60}")
        print(f"  {name}: {type(child).__name__} ({num_params:.1f}M params)")
        # Show sub-components for first transformer block
        if hasattr(child, '__getitem__'):
            print(f"    └─ {len(child)} blocks")
            if len(child) > 0:
                block = child[0]
                for sub_name, sub_child in block.named_children():
                    sub_params = sum(p.numel() for p in sub_child.parameters()) / 1e6
                    print(f"       ├─ {sub_name}: {type(sub_child).__name__} ({sub_params:.2f}M)")

    total_params = sum(p.numel() for p in transformer.parameters()) / 1e6
    print(f"\n{'=' * 80}")
    print(f"Total parameters: {total_params:.1f}M")
    print(f"\nKey differences from U-Net:")
    print(f"  - No skip connections (pure transformer)")
    print(f"  - No downsampling/upsampling paths")
    print(f"  - Patchification (image → tokens) instead of conv hierarchy")
    print(f"  - adaLN-Zero for conditioning instead of cross-attention")

In [ ]:
# Visualize patchification: how DiT converts images to token sequences
from torchvision.transforms.functional import to_tensor, to_pil_image

# Create a sample image (colored checkerboard for visibility)
img_size = 256
patch_size = 2  # DiT-XL/2 uses 2x2 patches on latent (but we demo on pixel space)
demo_patch_size = 32  # Larger patches for visualization

# Generate a sample image with clear structure
np.random.seed(42)
sample_img = np.zeros((img_size, img_size, 3), dtype=np.uint8)
# Create gradient + shapes
for i in range(img_size):
    for j in range(img_size):
        sample_img[i, j] = [int(255 * i / img_size), int(255 * j / img_size), 128]
# Add a circle
y, x = np.ogrid[:img_size, :img_size]
mask = (x - 128)**2 + (y - 128)**2 < 60**2
sample_img[mask] = [255, 255, 0]

fig, axes = plt.subplots(1, 4, figsize=(20, 5))

# 1. Original image
axes[0].imshow(sample_img)
axes[0].set_title(f"Original Image\n{img_size}x{img_size}x3", fontsize=12)
axes[0].axis("off")

# 2. Patch grid overlay
axes[1].imshow(sample_img)
n_patches = img_size // demo_patch_size
for i in range(n_patches + 1):
    axes[1].axhline(y=i * demo_patch_size, color='red', linewidth=0.5, alpha=0.7)
    axes[1].axvline(x=i * demo_patch_size, color='red', linewidth=0.5, alpha=0.7)
axes[1].set_title(f"Patch Grid\n{n_patches}x{n_patches} = {n_patches**2} patches", fontsize=12)
axes[1].axis("off")

# 3. Patches as sequence (show first 16 patches in a row)
patches = []
for i in range(n_patches):
    for j in range(n_patches):
        patch = sample_img[
            i * demo_patch_size:(i + 1) * demo_patch_size,
            j * demo_patch_size:(j + 1) * demo_patch_size
        ]
        patches.append(patch)

# Show first row of patches as a linear sequence
seq_display = np.concatenate(patches[:8], axis=1)
axes[2].imshow(seq_display)
for i in range(1, 8):
    axes[2].axvline(x=i * demo_patch_size, color='white', linewidth=2)
axes[2].set_title("Token Sequence (first 8 patches)\nPatch → Linear Projection → Token", fontsize=12)
axes[2].axis("off")

# 4. DiT actual patchification dimensions
axes[3].text(0.5, 0.9, "DiT-XL/2 Patchification", fontsize=14, fontweight='bold',
             ha='center', va='top', transform=axes[3].transAxes)
info_text = (
    "Input: 256x256x3 image\n"
    "→ VAE encode: 32x32x4 latent\n"
    "→ Patch (p=2): 16x16 patches\n"
    "→ Linear proj: 16x16 = 256 tokens\n"
    "→ Each token: 1152-dim (hidden_size)\n"
    "\n+ Positional embedding\n"
    "+ Timestep embedding\n"
    "+ Class label embedding\n"
    "\n→ 28 Transformer blocks\n"
    "→ Unpatchify to 32x32x4\n"
    "→ VAE decode: 256x256x3"
)
axes[3].text(0.5, 0.75, info_text, fontsize=10, ha='center', va='top',
             transform=axes[3].transAxes, family='monospace',
             bbox=dict(boxstyle='round', facecolor='lightyellow', alpha=0.8))
axes[3].axis("off")

plt.tight_layout()
plt.show()

In [ ]:
# Parameter count comparison: U-Net vs DiT variants
models = {
    "SD 1.5\nU-Net": 860,
    "DiT-XL/2\n(28 blocks)": 675,
    "DiT-L/2\n(24 blocks)": 458,
    "DiT-B/2\n(12 blocks)": 130,
    "DiT-S/2\n(12 blocks)": 33,
}

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Bar chart of param counts
colors = ['#e74c3c'] + ['#3498db'] * 4  # Red for U-Net, blue for DiT
bars = axes[0].bar(models.keys(), models.values(), color=colors, edgecolor='black', linewidth=0.5)
axes[0].set_ylabel("Parameters (M)", fontsize=12)
axes[0].set_title("Parameter Counts: U-Net vs DiT Variants", fontsize=13)
for bar, val in zip(bars, models.values()):
    axes[0].text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 15,
                 f"{val}M", ha='center', fontsize=10, fontweight='bold')
axes[0].set_ylim(0, 1000)

# Architecture comparison table
comparison = [
    ["Feature", "SD 1.5 U-Net", "DiT-XL/2"],
    ["Params", "860M", "675M"],
    ["Architecture", "U-Net (encoder-decoder)", "Transformer (isotropic)"],
    ["Skip connections", "Yes (essential)", "No"],
    ["Down/Upsampling", "Yes (4 levels)", "No"],
    ["Spatial processing", "Convolutions", "Self-attention + MLP"],
    ["Conditioning", "Cross-attention", "adaLN-Zero"],
    ["Input handling", "Full-res feature maps", "Patchified tokens"],
    ["Scaling behavior", "Diminishing returns", "Follows scaling laws"],
    ["FID-50K (256x256)", "~3.6 (cfg)", "2.27 (cfg=1.50)"],
]

table = axes[1].table(
    cellText=comparison[1:],
    colLabels=comparison[0],
    loc='center',
    cellLoc='center',
)
table.auto_set_font_size(False)
table.set_fontsize(9)
table.scale(1, 1.6)

# Color header
for j in range(3):
    table[0, j].set_facecolor('#34495e')
    table[0, j].set_text_props(color='white', fontweight='bold')

axes[1].axis("off")
axes[1].set_title("Architecture Comparison", fontsize=13)

plt.tight_layout()
plt.show()

## adaLN-Zero: How DiT Does Conditioning

Instead of cross-attention (SD's U-Net approach), DiT uses **adaptive layer normalization**:

1. Timestep `t` and class label `c` are embedded and summed
2. This embedding is projected to produce **scale (γ)** and **shift (β)** parameters for each LayerNorm
3. Each transformer block gets its own γ, β from the conditioning

The **"Zero"** part: the final linear projection in each block (that produces γ, β, α) is initialized to zero. This means at initialization, every DiT block acts as the **identity function** -- the model starts as a no-op and gradually learns to denoise. This stabilizes training significantly.

```
Standard LN:   y = (x - μ) / σ
adaLN:         y = γ(t,c) * (x - μ) / σ + β(t,c)
adaLN-Zero:    output = α(t,c) * block(adaLN(x))   # α initialized to 0
```

In [ ]:
# GPU: ~2 GB VRAM for training on MNIST
# ===========================================================================
# Flow Matching on MNIST -- Full Implementation
# ===========================================================================
# Key conceptual difference from DDPM:
#   DDPM: x_t = sqrt(alpha_bar_t) * x_0 + sqrt(1 - alpha_bar_t) * eps
#         model predicts eps (noise)
#   Flow: x_t = (1 - t) * x_0 + t * eps
#         model predicts v = eps - x_0 (velocity)
# That's it. The math change is ~20 lines of code.

# --- Data ---
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize([0.5], [0.5]),  # Scale to [-1, 1]
])
train_dataset = datasets.MNIST(root="./data", train=True, download=True, transform=transform)
train_loader = DataLoader(train_dataset, batch_size=128, shuffle=True, num_workers=2, drop_last=True)

# --- Sinusoidal time embedding ---
class SinusoidalPosEmb(nn.Module):
    def __init__(self, dim: int):
        super().__init__()
        self.dim = dim

    def forward(self, t: torch.Tensor) -> torch.Tensor:
        half_dim = self.dim // 2
        emb = np.log(10000) / (half_dim - 1)
        emb = torch.exp(torch.arange(half_dim, device=t.device) * -emb)
        emb = t[:, None] * emb[None, :]
        return torch.cat([emb.sin(), emb.cos()], dim=-1)

# --- Residual block with time conditioning ---
class ResBlock(nn.Module):
    def __init__(self, in_ch: int, out_ch: int, time_dim: int):
        super().__init__()
        self.conv1 = nn.Conv2d(in_ch, out_ch, 3, padding=1)
        self.conv2 = nn.Conv2d(out_ch, out_ch, 3, padding=1)
        self.time_mlp = nn.Linear(time_dim, out_ch)
        self.norm1 = nn.GroupNorm(8, out_ch)
        self.norm2 = nn.GroupNorm(8, out_ch)
        self.skip = nn.Conv2d(in_ch, out_ch, 1) if in_ch != out_ch else nn.Identity()

    def forward(self, x: torch.Tensor, t_emb: torch.Tensor) -> torch.Tensor:
        h = self.norm1(F.silu(self.conv1(x)))
        h = h + self.time_mlp(F.silu(t_emb))[:, :, None, None]
        h = self.norm2(F.silu(self.conv2(h)))
        return h + self.skip(x)

# --- Simple U-Net for MNIST (same arch works for both DDPM and flow matching) ---
class SimpleUNet(nn.Module):
    """Small U-Net for 28x28 grayscale. Predicts either noise (DDPM) or velocity (flow)."""
    def __init__(self, in_ch: int = 1, base_ch: int = 64, time_dim: int = 256):
        super().__init__()
        self.time_embed = nn.Sequential(
            SinusoidalPosEmb(time_dim),
            nn.Linear(time_dim, time_dim),
            nn.SiLU(),
            nn.Linear(time_dim, time_dim),
        )
        # Encoder
        self.enc1 = ResBlock(in_ch, base_ch, time_dim)
        self.enc2 = ResBlock(base_ch, base_ch * 2, time_dim)
        self.down1 = nn.Conv2d(base_ch, base_ch, 3, stride=2, padding=1)
        self.down2 = nn.Conv2d(base_ch * 2, base_ch * 2, 3, stride=2, padding=1)
        # Bottleneck
        self.bot = ResBlock(base_ch * 2, base_ch * 2, time_dim)
        # Decoder
        self.up2 = nn.ConvTranspose2d(base_ch * 2, base_ch * 2, 4, stride=2, padding=1)
        self.dec2 = ResBlock(base_ch * 4, base_ch, time_dim)  # concat skip
        self.up1 = nn.ConvTranspose2d(base_ch, base_ch, 4, stride=2, padding=1)
        self.dec1 = ResBlock(base_ch * 2, base_ch, time_dim)  # concat skip
        self.out_conv = nn.Conv2d(base_ch, in_ch, 1)

    def forward(self, x: torch.Tensor, t: torch.Tensor) -> torch.Tensor:
        t_emb = self.time_embed(t)
        # Encoder
        h1 = self.enc1(x, t_emb)            # 28x28
        h = self.down1(h1)                   # 14x14
        h2 = self.enc2(h, t_emb)             # 14x14
        h = self.down2(h2)                   # 7x7
        # Bottleneck
        h = self.bot(h, t_emb)               # 7x7
        # Decoder
        h = self.up2(h)                      # 14x14
        h = torch.cat([h, h2], dim=1)        # skip connection
        h = self.dec2(h, t_emb)              # 14x14
        h = self.up1(h)                      # 28x28
        h = torch.cat([h, h1], dim=1)        # skip connection
        h = self.dec1(h, t_emb)              # 28x28
        return self.out_conv(h)


class FlowMatchingTrainer:
    """Flow matching training -- the code diff from DDPM is ~20 lines."""

    def __init__(self, model: nn.Module, lr: float = 1e-4):
        self.model = model
        self.optimizer = torch.optim.AdamW(model.parameters(), lr=lr)

    def get_train_tuple(self, x_0: torch.Tensor):
        """Core of flow matching -- compare with DDPM's noising."""
        t = torch.rand(x_0.shape[0], device=x_0.device)  # Uniform [0, 1] -- no schedule!
        noise = torch.randn_like(x_0)
        # Linear interpolation (not a noise schedule!)
        x_t = (1 - t.view(-1, 1, 1, 1)) * x_0 + t.view(-1, 1, 1, 1) * noise
        # Target is the velocity field: v = dx/dt = noise - x_0
        target = noise - x_0
        return x_t, t, target

    def train_step(self, x_0: torch.Tensor) -> float:
        self.optimizer.zero_grad()
        x_t, t, target = self.get_train_tuple(x_0)
        pred = self.model(x_t, t)
        loss = F.mse_loss(pred, target)
        loss.backward()
        self.optimizer.step()
        return loss.item()


class DDPMTrainer:
    """DDPM training for comparison."""

    def __init__(self, model: nn.Module, n_steps: int = 1000, lr: float = 1e-4):
        self.model = model
        self.n_steps = n_steps
        self.optimizer = torch.optim.AdamW(model.parameters(), lr=lr)
        # Noise schedule
        betas = torch.linspace(1e-4, 0.02, n_steps)
        alphas = 1.0 - betas
        self.alpha_bar = torch.cumprod(alphas, dim=0).to(device)

    def get_train_tuple(self, x_0: torch.Tensor):
        t = torch.randint(0, self.n_steps, (x_0.shape[0],), device=x_0.device)
        noise = torch.randn_like(x_0)
        alpha_bar_t = self.alpha_bar[t].view(-1, 1, 1, 1)
        x_t = torch.sqrt(alpha_bar_t) * x_0 + torch.sqrt(1 - alpha_bar_t) * noise
        return x_t, t.float() / self.n_steps, noise  # normalize t to [0,1]

    def train_step(self, x_0: torch.Tensor) -> float:
        self.optimizer.zero_grad()
        x_t, t, target = self.get_train_tuple(x_0)
        pred = self.model(x_t, t)
        loss = F.mse_loss(pred, target)
        loss.backward()
        self.optimizer.step()
        return loss.item()


# Instantiate both models
flow_model = SimpleUNet(in_ch=1, base_ch=64).to(device)
ddpm_model = SimpleUNet(in_ch=1, base_ch=64).to(device)

flow_params = sum(p.numel() for p in flow_model.parameters()) / 1e6
print(f"Model parameters: {flow_params:.2f}M (same architecture for both)")
print(f"\nFlow matching trainer: predicts velocity v = noise - x_0")
print(f"DDPM trainer: predicts noise eps")
print(f"Same model, same capacity -- only the training objective differs.")

In [ ]:
# GPU: ~2 GB VRAM, ~30 min on RTX 2080
# Train both models side by side

N_EPOCHS = 20
flow_trainer = FlowMatchingTrainer(flow_model, lr=2e-4)
ddpm_trainer = DDPMTrainer(ddpm_model, n_steps=1000, lr=2e-4)

flow_losses = []
ddpm_losses = []

for epoch in range(N_EPOCHS):
    flow_epoch_loss = []
    ddpm_epoch_loss = []

    pbar = tqdm(train_loader, desc=f"Epoch {epoch + 1}/{N_EPOCHS}")
    for batch_idx, (x, _) in enumerate(pbar):
        x = x.to(device)

        flow_loss = flow_trainer.train_step(x)
        ddpm_loss = ddpm_trainer.train_step(x)

        flow_epoch_loss.append(flow_loss)
        ddpm_epoch_loss.append(ddpm_loss)

        if batch_idx % 50 == 0:
            pbar.set_postfix({
                "flow_loss": f"{flow_loss:.4f}",
                "ddpm_loss": f"{ddpm_loss:.4f}",
            })

    flow_losses.append(np.mean(flow_epoch_loss))
    ddpm_losses.append(np.mean(ddpm_epoch_loss))
    print(f"Epoch {epoch + 1}: Flow={flow_losses[-1]:.4f}, DDPM={ddpm_losses[-1]:.4f}")

# Plot training curves
fig, ax = plt.subplots(1, 1, figsize=(10, 5))
ax.plot(flow_losses, label="Flow Matching (velocity)", linewidth=2, color='#e74c3c')
ax.plot(ddpm_losses, label="DDPM (noise)", linewidth=2, color='#3498db')
ax.set_xlabel("Epoch", fontsize=12)
ax.set_ylabel("MSE Loss", fontsize=12)
ax.set_title("Training Loss: Flow Matching vs DDPM", fontsize=14)
ax.legend(fontsize=12)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Flow matching sampling: Euler ODE integration

@torch.no_grad()
def sample_flow(model: nn.Module, n_samples: int, n_steps: int = 50,
                return_trajectory: bool = False) -> torch.Tensor:
    """ODE sampling: start at noise (t=1), integrate to data (t=0)."""
    model.eval()
    x = torch.randn(n_samples, 1, 28, 28, device=device)
    dt = 1.0 / n_steps
    trajectory = [x.clone()] if return_trajectory else None

    for i in range(n_steps):
        t_val = 1.0 - i * dt
        t = torch.ones(n_samples, device=device) * t_val
        v = model(x, t)  # predicted velocity
        x = x - v * dt   # Euler step: x_{t-dt} = x_t - v * dt
        if return_trajectory:
            trajectory.append(x.clone())

    model.train()
    if return_trajectory:
        return x, trajectory
    return x


@torch.no_grad()
def sample_ddpm(model: nn.Module, n_samples: int, n_steps: int = 50,
                total_steps: int = 1000,
                return_trajectory: bool = False) -> torch.Tensor:
    """DDPM sampling with stride (fewer steps than training steps)."""
    model.eval()
    betas = torch.linspace(1e-4, 0.02, total_steps, device=device)
    alphas = 1.0 - betas
    alpha_bar = torch.cumprod(alphas, dim=0)

    # Use strided timesteps for fewer-step sampling
    stride = total_steps // n_steps
    timesteps = list(range(0, total_steps, stride))[::-1]  # Descending

    x = torch.randn(n_samples, 1, 28, 28, device=device)
    trajectory = [x.clone()] if return_trajectory else None

    for i, t_val in enumerate(timesteps):
        t = torch.ones(n_samples, device=device) * t_val / total_steps
        pred_noise = model(x, t)

        ab_t = alpha_bar[t_val]
        a_t = alphas[t_val]
        b_t = betas[t_val]

        # DDPM update
        x = (1 / torch.sqrt(a_t)) * (x - (b_t / torch.sqrt(1 - ab_t)) * pred_noise)
        if t_val > 0:
            x = x + torch.sqrt(b_t) * torch.randn_like(x)

        if return_trajectory:
            trajectory.append(x.clone())

    model.train()
    if return_trajectory:
        return x, trajectory
    return x


# Generate and display samples from flow matching
flow_samples = sample_flow(flow_model, n_samples=64, n_steps=50)
flow_samples = (flow_samples.clamp(-1, 1) + 1) / 2  # rescale to [0,1]

grid = make_grid(flow_samples, nrow=8, padding=2)
plt.figure(figsize=(10, 10))
plt.imshow(grid.permute(1, 2, 0).cpu(), cmap='gray')
plt.title("Flow Matching Samples (50 Euler steps)", fontsize=14)
plt.axis("off")
plt.show()

In [ ]:
# Side-by-side comparison: DDPM vs Flow Matching at different step counts

step_counts = [5, 10, 20, 50]
n_show = 8  # samples per setting

# Fix the seed for fair comparison
torch.manual_seed(42)
base_noise_flow = torch.randn(n_show, 1, 28, 28, device=device)
torch.manual_seed(42)
base_noise_ddpm = torch.randn(n_show, 1, 28, 28, device=device)

fig, axes = plt.subplots(2, len(step_counts), figsize=(4 * len(step_counts), 8))

for col, n_steps in enumerate(step_counts):
    # Flow matching samples
    flow_model.eval()
    with torch.no_grad():
        x_flow = base_noise_flow.clone()
        dt = 1.0 / n_steps
        for i in range(n_steps):
            t_val = 1.0 - i * dt
            t = torch.ones(n_show, device=device) * t_val
            v = flow_model(x_flow, t)
            x_flow = x_flow - v * dt

    x_flow = (x_flow.clamp(-1, 1) + 1) / 2
    grid_flow = make_grid(x_flow, nrow=4, padding=1)
    axes[0, col].imshow(grid_flow.permute(1, 2, 0).cpu(), cmap='gray')
    axes[0, col].set_title(f"{n_steps} steps", fontsize=12)
    axes[0, col].axis("off")
    if col == 0:
        axes[0, col].set_ylabel("Flow Matching", fontsize=14, fontweight='bold')

    # DDPM samples
    ddpm_model.eval()
    with torch.no_grad():
        x_ddpm = sample_ddpm(ddpm_model, n_show, n_steps=n_steps)

    x_ddpm = (x_ddpm.clamp(-1, 1) + 1) / 2
    grid_ddpm = make_grid(x_ddpm, nrow=4, padding=1)
    axes[1, col].imshow(grid_ddpm.permute(1, 2, 0).cpu(), cmap='gray')
    axes[1, col].axis("off")
    if col == 0:
        axes[1, col].set_ylabel("DDPM", fontsize=14, fontweight='bold')

fig.suptitle("Quality vs Number of Sampling Steps", fontsize=16, y=1.02)
plt.tight_layout()
plt.show()
print("Flow matching should maintain quality at lower step counts (straighter trajectories).")
print("DDPM degrades faster because curved trajectories need more discretization steps.")

In [ ]:
# Visualize sampling trajectories: DDPM vs Flow Matching
# Project denoising paths into 2D using PCA

from sklearn.decomposition import PCA

n_traj_steps = 50

# Get trajectories for a single sample
_, flow_traj = sample_flow(flow_model, n_samples=1, n_steps=n_traj_steps, return_trajectory=True)
_, ddpm_traj = sample_ddpm(ddpm_model, n_samples=1, n_steps=n_traj_steps, return_trajectory=True)

# Flatten trajectories to vectors
flow_flat = torch.stack([x.flatten() for x in flow_traj]).cpu().numpy()  # (steps+1, 784)
ddpm_flat = torch.stack([x.flatten() for x in ddpm_traj]).cpu().numpy()

# Fit PCA on combined trajectories
combined = np.concatenate([flow_flat, ddpm_flat], axis=0)
pca = PCA(n_components=2)
combined_2d = pca.fit_transform(combined)

flow_2d = combined_2d[:len(flow_flat)]
ddpm_2d = combined_2d[len(flow_flat):]

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Plot Flow Matching trajectory
axes[0].plot(flow_2d[:, 0], flow_2d[:, 1], 'o-', color='#e74c3c', markersize=3,
             linewidth=1.5, alpha=0.7, label='Flow Matching')
axes[0].plot(flow_2d[0, 0], flow_2d[0, 1], 's', color='#e74c3c', markersize=12, label='Start (noise)')
axes[0].plot(flow_2d[-1, 0], flow_2d[-1, 1], '*', color='#e74c3c', markersize=15, label='End (data)')
axes[0].set_title("Flow Matching Trajectory\n(should be ~straight)", fontsize=12)
axes[0].legend(fontsize=9)
axes[0].grid(True, alpha=0.3)

# Plot DDPM trajectory
axes[1].plot(ddpm_2d[:, 0], ddpm_2d[:, 1], 'o-', color='#3498db', markersize=3,
             linewidth=1.5, alpha=0.7, label='DDPM')
axes[1].plot(ddpm_2d[0, 0], ddpm_2d[0, 1], 's', color='#3498db', markersize=12, label='Start (noise)')
axes[1].plot(ddpm_2d[-1, 0], ddpm_2d[-1, 1], '*', color='#3498db', markersize=15, label='End (data)')
axes[1].set_title("DDPM Trajectory\n(should be curved)", fontsize=12)
axes[1].legend(fontsize=9)
axes[1].grid(True, alpha=0.3)

# Overlay comparison
axes[2].plot(flow_2d[:, 0], flow_2d[:, 1], 'o-', color='#e74c3c', markersize=2,
             linewidth=1.5, alpha=0.6, label='Flow Matching')
axes[2].plot(ddpm_2d[:, 0], ddpm_2d[:, 1], 'o-', color='#3498db', markersize=2,
             linewidth=1.5, alpha=0.6, label='DDPM')
axes[2].set_title("Overlay\n(straighter = fewer steps needed)", fontsize=12)
axes[2].legend(fontsize=10)
axes[2].grid(True, alpha=0.3)

# Compute straightness metric: ratio of endpoint distance to total path length
def straightness(traj_2d):
    endpoint_dist = np.linalg.norm(traj_2d[-1] - traj_2d[0])
    path_length = sum(np.linalg.norm(traj_2d[i+1] - traj_2d[i]) for i in range(len(traj_2d) - 1))
    return endpoint_dist / (path_length + 1e-8)

print(f"Straightness (1.0 = perfectly straight):")
print(f"  Flow Matching: {straightness(flow_2d):.3f}")
print(f"  DDPM:          {straightness(ddpm_2d):.3f}")

plt.suptitle("Sampling Trajectories in PCA Space", fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

## Checkpoint

**Flow matching vs DDPM explained to a colleague:**

Same goal (generate data from noise), different math.

- **DDPM:** Adds noise according to a fixed schedule, learns to predict/remove the noise at each step. The forward process follows a specific variance schedule, and the reverse process must follow the same curved path backward.
- **Flow matching:** Defines a *straight* path between noise and data via linear interpolation `x_t = (1-t)*x_0 + t*noise`, and learns the velocity `v = noise - x_0` to travel that path.

**Result:** Flow matching needs fewer sampling steps because the paths are straighter (less discretization error with Euler integration). This is why modern models (SD3, Flux, Wan2.1) use flow matching or its close relative, rectified flow.

**Why DiT matters:** The transformer backbone (DiT) replaces U-Net's conv hierarchy with a flat sequence of transformer blocks over patchified tokens. This is simpler, scales predictably (follows LLM scaling laws), and works naturally with flow matching.

**Further reading:**
- DiT: [2212.09748](https://arxiv.org/abs/2212.09748)
- Flow Matching for Generative Modeling: [2210.02747](https://arxiv.org/abs/2210.02747)
- Rectified Flow: [2209.03003](https://arxiv.org/abs/2209.03003)